In [1]:
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt
import arviz as az

import joblib

In [2]:
gk_betas = joblib.load('../estimates/league_models_gk')
def_betas = joblib.load('../estimates/league_models_def')
mid_betas = joblib.load('../estimates/league_models_mid')
fwd_betas = joblib.load('../estimates/league_models_fwd')

In [3]:
player_data = pd.read_csv('../../rolled_data_24_25.csv')
player_data

,Unnamed: 0,element,round,fixture,opponent_team,total_points,was_home,team_h_score,team_a_score,minutes,...,goals_prevented_per_90_5_ewm,sweeper_actions_per_90_1_ewm,sweeper_actions_per_90_3_ewm,sweeper_actions_per_90_5_ewm,tackles_won_percent_per_90_1_ewm,tackles_won_percent_per_90_3_ewm,tackles_won_percent_per_90_5_ewm,select_count,league_select_perc,league_select_probs
0,0,1,1,2.0,Wolves,0.0,True,2.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
1,1,1,2,11.0,Aston Villa,0.0,False,0.0,2.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,1,3,21.0,Brighton,0.0,True,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3,1,4,39.0,Spurs,0.0,False,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4,1,5,47.0,Man City,0.0,False,2.0,2.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26914,26914,801,37,361.0,Newcastle,0.0,True,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
26915,26915,801,38,378.0,Southampton,0.0,False,1.0,2.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
26916,26916,802,38,371.0,Bournemouth,0.0,False,2.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
26917,26917,803,38,376.0,Newcastle,0.0,False,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
league_selections = pd.read_csv('../../league_selections_df.csv')
league_selections

,team_id,round,squad,midfielders,forwards,defenders,goalkeepers,goalkeeper_binary,defender_binary,midfielder_binary,forward_binary
0,205,1,"[201, 350, 231, 333, 328, 317, 181, 19, 351, 4...","[328, 317, 181, 19, 481]","[351, 401, 82]","[350, 231, 333, 255, 461]","[201, 209]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ..."
1,205,2,"[201, 350, 255, 461, 328, 317, 181, 19, 351, 4...","[328, 317, 181, 19, 481]","[351, 401, 251]","[350, 255, 461, 333, 231]","[201, 209]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,205,3,"[201, 350, 255, 231, 328, 317, 19, 54, 351, 40...","[328, 317, 19, 54, 481]","[351, 401, 251]","[350, 255, 231, 333, 461]","[201, 209]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,205,4,"[201, 311, 350, 255, 328, 317, 19, 54, 351, 40...","[328, 317, 19, 54, 481]","[351, 401, 251]","[311, 350, 255, 231, 461]","[201, 209]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,205,5,"[201, 461, 350, 255, 311, 19, 328, 54, 251, 35...","[19, 328, 54, 317, 481]","[251, 351, 58]","[461, 350, 255, 311, 231]","[201, 209]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
...,...,...,...,...,...,...,...,...,...,...,...
755,4131447,34,"[185, 533, 418, 579, 163, 327, 182, 328, 252, ...","[327, 182, 328, 392, 247]","[252, 401, 541]","[533, 418, 579, 163, 70]","[185, 152]","[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, ...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
756,4131447,35,"[15, 44, 350, 211, 99, 328, 199, 54, 447, 755,...","[99, 328, 199, 54, 762]","[447, 755, 207]","[44, 350, 211, 18, 361]","[15, 513]","[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
757,4131447,36,"[513, 361, 211, 350, 99, 328, 199, 106, 447, 1...","[99, 328, 199, 106, 54]","[447, 110, 207]","[361, 211, 350, 44, 18]","[513, 15]","[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, ..."
758,4131447,37,"[513, 361, 211, 350, 99, 106, 199, 328, 54, 11...","[99, 106, 199, 328, 54]","[110, 58, 447]","[361, 211, 350, 44, 18]","[513, 15]","[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, ..."


In [5]:
import random

managers = [
    205, 901, 4213, 7497, 13603,
    23881, 38054, 38351, 69718, 152245,
    198971, 577273, 593532, 696127, 749244,
    1222335, 1441046, 1546929, 2634457, 4131447
]


def create_fixtures(managers, seed=None):
    """Create a randomized single round-robin fixture."""

    if seed is not None:
        random.seed(seed)

    managers = managers.copy()
    random.shuffle(managers)

    fixed = managers[0]
    rotating = managers[1:]

    fixtures = {}

    for gw in range(1, len(managers)):
        current = [fixed] + rotating

        gw_fixtures = {}

        for i in range(len(current) // 2):
            manager1 = current[i]
            manager2 = current[-(i + 1)]

            gw_fixtures[manager1] = manager2
            gw_fixtures[manager2] = manager1

        fixtures[gw] = gw_fixtures

        # Rotate all except the fixed manager
        rotating = [rotating[-1]] + rotating[:-1]

    return fixtures


def get_opponent(manager_id, gameweek, fixtures):
    """Return the opponent of a manager in a particular GW."""

    if gameweek not in fixtures:
        raise ValueError(f"Gameweek must be between 1 and 19")

    if manager_id not in fixtures[gameweek]:
        raise ValueError(f"Manager {manager_id} not found")

    return fixtures[gameweek][manager_id]


# --------------------------------------------------
# Generate randomized league
# --------------------------------------------------

fixtures_leg_1 = create_fixtures(managers, seed=61)
manager_id = 205

opponents_by_round_leg_1 = {gw: get_opponent(manager_id, gw, fixtures_leg_1) for gw in range(1, 20)}


fixtures_leg_2 = create_fixtures(managers, seed=16)
manager_id = 205

opponents_by_round_leg_2 = {gw+19: get_opponent(manager_id, gw, fixtures_leg_2) for gw in range(1, 20)}

opponents_by_round = opponents_by_round_leg_1 |  opponents_by_round_leg_2
opponents_by_round

{1: 38351,
 2: 69718,
 3: 7497,
 4: 4213,
 5: 901,
 6: 23881,
 7: 38054,
 8: 13603,
 9: 593532,
 10: 749244,
 11: 696127,
 12: 4131447,
 13: 152245,
 14: 1441046,
 15: 1222335,
 16: 1546929,
 17: 198971,
 18: 2634457,
 19: 577273,
 20: 69718,
 21: 4213,
 22: 1441046,
 23: 4131447,
 24: 1222335,
 25: 152245,
 26: 7497,
 27: 901,
 28: 198971,
 29: 749244,
 30: 593532,
 31: 23881,
 32: 1546929,
 33: 577273,
 34: 2634457,
 35: 696127,
 36: 38351,
 37: 38054,
 38: 13603}

## The Longitudinal Individual Update (the posterior)

For a target manager $m$ and position $j$, we treat the league-wide fit $\beta_{j,t}^{\text{league}}$ (from `gk_betas`/`def_betas`/`mid_betas`/`fwd_betas`, keyed by gameweek) at a single reference gameweek as the informative prior, and fit only $\boldsymbol{\theta}_{j,m} = \{\Delta\beta_{j,m}, \gamma_m\}$ against the manager's own selection history $t = 1, \ldots, T-1$ (eq. `Eq:posterior_manager`).

Since the eligible player pool for a position grows week to week (promotions, new signings), each week's occupancy vector $Y_{m,t,\cdot}$ has a different length, so the per-week Dirichlet log-density in eq. `Eq:alpha_m` / `Eq:posterior_manager` is accumulated manually via `pm.Potential` rather than a single vectorized `pm.Dirichlet`.


In [15]:
import ast

LEAGUE_FITS = {
    'Goalkeeper': gk_betas,
    'Defender': def_betas,
    'Midfielder': mid_betas,
    'Forward': fwd_betas,
}

POSITION_BINARY_COL = {
    'Goalkeeper': 'goalkeeper_binary',
    'Defender': 'defender_binary',
    'Midfielder': 'midfielder_binary',
    'Forward': 'forward_binary',
}


def get_league_fit_for_round(league_fits, target_round):
    """
    Pick the league fit (dict from fit_position_pipeline) at `target_round`, or -- since
    league models were only fit from gameweek 6 onward once there was enough data -- the
    most recent fit available at or before it. This single fit supplies both beta^league
    and the fixed feature/standardization basis used across the *entire* manager history,
    per eq:posterior_manager (beta_league there carries no t-subscript).
    """
    available = [gw for gw in league_fits if gw <= target_round]
    if not available:
        raise ValueError(
            f"No league fit available at or before round {target_round} "
            f"(earliest available is {min(league_fits)})"
        )
    return league_fits[max(available)]


def _position_pool(player_data, position, rnd, feats):
    """
    The player pool for `position` at gameweek `rnd`, sorted by element id ascending --
    this ordering matches the *_binary columns in league_selections for that round.
    """
    pool = player_data[(player_data['round'] == rnd) & (player_data['position'] == position)] \
        .sort_values('element')
    elements = pool['element'].to_numpy()
    # league models are trained on rounds > 3 with NaNs (unfilled EWM lookback windows
    # early in the season) filled to 0 -- match that convention here (see beta_mid.ipynb etc).

    X_raw = pool[feats].fillna(0).to_numpy(dtype=float)
    return elements, X_raw

In [16]:
def build_manager_history_arrays(team_id, position, league_fit, player_data, league_selections,
                                  target_round, eps=1e-6, min_round=4):
    """
    Assemble the manager's selection history t = 1, ..., T-1 (T = target_round) for one
    position: per week, the standardized feature matrix X_t (fixed feats/mean/std from
    `league_fit`), the retention lag Y_{m,t-1,k}, and the eps-smoothed occupancy vector
    p_{j,m,t} used as the observed Dirichlet outcome (eq:alpha_m / eq:posterior_manager).
    Y_{m,0,k} is taken to be all-zero (no prior ownership before the manager's first
    recorded gameweek).

    `min_round` mirrors the league fit's own training window (rounds > 3, once the EWM
    features have enough lookback games) -- weeks before it are excluded rather than fed
    in with degenerate/NaN-derived features.
    """
    bin_col = POSITION_BINARY_COL[position]
    feats = league_fit['feats']
    X_mean, X_std = league_fit['X_mean'], league_fit['X_std']

    rows = league_selections[
        (league_selections['team_id'] == team_id)
        & (league_selections['round'] < target_round)
        & (league_selections['round'] >= min_round)
    ].sort_values('round')

    history = []
    prev_owned = {}
    for _, row in rows.iterrows():
        rnd = int(row['round'])
        elements, X_raw = _position_pool(player_data, position, rnd, feats)
        if len(elements) == 0:
            continue

        raw_bin = row[bin_col]
        y_t = np.asarray(ast.literal_eval(raw_bin) if isinstance(raw_bin, str) else raw_bin, dtype=float)
        if len(y_t) != len(elements):
            # occasionally a gameweek's selection export doesn't line up with the player
            # pool snapshot for that round (e.g. incomplete data for a blank/postponed
            # fixture) -- skip that single week rather than fail the whole history.
            print(f"  skipping round {rnd} for team {team_id} -- pool/selection length mismatch "
                  f"({len(elements)} vs {len(y_t)}), likely incomplete gameweek data")
            continue

        X_t = (X_raw - X_mean) / X_std
        p_t = y_t + eps
        p_t = p_t / p_t.sum()
        y_prev = np.array([prev_owned.get(e, 0.0) for e in elements])

        history.append({"round": rnd, "elements": elements, "X": X_t, "p": p_t, "y_prev": y_prev})
        prev_owned = dict(zip(elements.tolist(), y_t.tolist()))

    return history

In [ ]:
def fit_manager_pipeline(team_id, position, league_fit, player_data, league_selections, target_round,
                          delta_sigma=1.0, gamma_sigma=1.0, eps=1e-6,
                          draws=2000, tune=2000, chains=4, cores=4,
                          target_accept=0.95, max_treedepth=12, random_seed=None):
    """
    Fit theta_{j,m} = {Delta_beta_m, gamma_m} for one manager/position, per
    eq:posterior_manager, treating `league_fit`'s beta_mean as the fixed informative
    prior mean (beta^league) and standardizing every week's raw features on
    `league_fit`'s training X_mean/X_std so Delta_beta_m lives on the same scale.

    alpha_{j,m,t,k} = exp( (beta^league + Delta_beta_m) . X_{t,k} + gamma_m * Y_{m,t-1,k} )   (eq:alpha_m)

    Regularizing priors: Delta_beta_m ~ Normal(0, delta_sigma) (zero-mean, i.e. "no
    deviation from the league" is the default), gamma_m ~ HalfNormal(gamma_sigma)
    (transfer friction only ever inflates retained players' utility, never deflates it).
    """
    history = build_manager_history_arrays(
        team_id, position, league_fit, player_data, league_selections, target_round, eps=eps
    )
    if len(history) == 0:
        raise ValueError(f"No selection history before round {target_round} for team {team_id}, position {position!r}")

    beta_league = league_fit['beta_mean']
    n_features = len(league_fit['feats'])

    with pm.Model() as model:
        delta_beta = pm.Normal("delta_beta", mu=0.0, sigma=delta_sigma, shape=n_features)
        gamma_m = pm.HalfNormal("gamma_m", sigma=gamma_sigma)

        beta_m = beta_league + delta_beta

        loglik_terms = []
        for week in history:
            X_t = pt.as_tensor_variable(week["X"])
            p_t = pt.as_tensor_variable(week["p"])
            y_prev = pt.as_tensor_variable(week["y_prev"])

            linear = pt.dot(X_t, beta_m) + gamma_m * y_prev
            alpha_t = pt.clip(pt.exp(pt.clip(linear, -30, 30)), 1e-6, 1e6)

            # Dirichlet(p_t; alpha_t) log-density, written out explicitly (rather than
            # pm.Dirichlet) since alpha_t's length varies week to week with the pool size.
            loglik_t = (
                pt.gammaln(pt.sum(alpha_t)) - pt.sum(pt.gammaln(alpha_t))
                + pt.sum((alpha_t - 1.0) * pt.log(p_t))
            )

            loglik_terms.append(loglik_t)

        pm.Potential("manager_loglik", pt.sum(pt.stack(loglik_terms)))

        trace = pm.sample(
            draws, tune=tune, chains=chains, cores=cores,
            target_accept=target_accept, max_treedepth=max_treedepth,
            random_seed=random_seed,
        )

    pm.model_to_graphviz(model)

    summary = az.summary(trace, var_names=["delta_beta", "gamma_m"])
    n_div = int(trace.sample_stats["diverging"].sum())
    print(f"  team {team_id} / {position}: {len(history)} weeks of history, divergences: {n_div}")
    if n_div > 0:
        print("  WARNING: divergences present -- treat this fit as provisional")

    return {
        "team_id": team_id,
        "position": position,
        "target_round": target_round,
        "feats": league_fit['feats'],
        "n_weeks": len(history),
        "trace": trace,
        "summary": summary,
        "delta_beta_mean": trace.posterior["delta_beta"].mean(dim=("chain", "draw")).values,
        "gamma_mean": float(trace.posterior["gamma_m"].mean()),
    }

def alpha_manager(manager_fit, league_fit, X_new_raw, y_prev):
    """
    Apply a fitted manager model (from fit_manager_pipeline) to a new week's raw
    features and retention lag Y_{m,t-1,k}, per eq:alpha_m.
    """
    X_std_new = (X_new_raw - league_fit["X_mean"]) / league_fit["X_std"]
    beta_m = league_fit["beta_mean"] + manager_fit["delta_beta_mean"]
    linear = X_std_new @ beta_m + manager_fit["gamma_mean"] * y_prev
    linear = np.clip(linear, -30, 30)
    return np.exp(linear)

In [34]:
for round in range(6, 39):
    man_fits = {}
    print(f"Fitting manager models for round {round}...")

    for position in ['Goalkeeper', 'Defender', 'Midfielder', 'Forward']:
        # Demo: fit one manager's midfield deviation from the league baseline, using their
        # selection history up to (but not including) round.
        team_id = opponents_by_round[round]
        target_round = round
        league_fit = get_league_fit_for_round(LEAGUE_FITS[position], target_round)

        manager_fit = fit_manager_pipeline(
            team_id, position, league_fit, player_data, league_selections, target_round,
            random_seed=42,
        )

        man_fits[ position] = manager_fit#["summary"]
    joblib.dump({round: man_fits}, f'./estimates/manager_fits_{round}.joblib')

Initializing NUTS using jitter+adapt_diag...


Fitting manager models for round 6...


Multiprocess sampling (4 chains in 4 jobs)
NUTS: [delta_beta, gamma_m]


Output()

Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 104 seconds.
Initializing NUTS using jitter+adapt_diag...


...

  team 13603 / Midfielder: 34 weeks of history, divergences: 0


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [delta_beta, gamma_m]


Output()

Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 70 seconds.


  team 13603 / Forward: 34 weeks of history, divergences: 0


In [22]:
manager_fits_7 = joblib.load('./estimates/manager_fits_7.joblib')
manager_fits_7

{7: {'Goalkeeper': {'team_id': 38054,
   'position': 'Goalkeeper',
   'target_round': 7,
   'feats': ['transfers_in',
    'total_points_per_90_rolling_5',
    'clean_sheets_per_90_rolling_1',
    'expected_goal_involvements_3_ewm',
    'tackles_5_ewm',
    'sweeper_actions_per_90_5_ewm'],
   'n_weeks': 3,
   'trace': Inference data with groups:
   	> posterior
   	> sample_stats,
   'summary':                  mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  ess_bulk  \
   delta_beta[0]  -0.607  0.039  -0.681   -0.537      0.001    0.000    3790.0   
   delta_beta[1]   0.403  0.033   0.341    0.464      0.001    0.000    3885.0   
   delta_beta[2]   0.276  0.040   0.199    0.349      0.001    0.000    3859.0   
   delta_beta[3]  -0.151  0.038  -0.224   -0.080      0.001    0.000    4561.0   
   delta_beta[4]  -0.122  0.024  -0.169   -0.077      0.000    0.000    2749.0   
   delta_beta[5]   0.202  0.038   0.133    0.276      0.001    0.000    3845.0   
   gamma_m        14.118  0.096 

### Applying `alpha_manager`: calibration check for gameweek 7

For each position, compute `alpha_{j,m,7,k}` (eq. `Eq:alpha_m`) for every player in that
week's pool using the fitted `manager_fits_7` models, normalize to `p_{j,m,7,k}`, and
check where the manager's actually-owned players land in that ranking.


In [ ]:
def evaluate_manager_fit(team_id, position, manager_fit, league_fit, player_data, league_selections, gw):
    """
    Apply a fitted manager model to gameweek `gw` itself: compute alpha_{j,m,gw,k}
    (eq:alpha_m) for every player in that week's pool via alpha_manager, normalize to
    p_{j,m,gw,k}, and rank against who the manager actually owned that week.

    Returns (result, owned_ranks): `result` is the full ranked DataFrame (columns:
    element, owned_gw{gw}, owned_prev, p_manager), and `owned_ranks` is the 1-indexed
    rank of each actually-owned player in that ranking (out of len(result)).
    """
    elements, X_raw = _position_pool(player_data, position, gw, league_fit['feats'])

    row = league_selections[
        (league_selections['team_id'] == team_id) & (league_selections['round'] == gw)
    ].iloc[0]
    y_actual = np.asarray(ast.literal_eval(row[POSITION_BINARY_COL[position]]), dtype=float)
    assert len(y_actual) == len(elements), f"pool/selection mismatch for {position} at round {gw}"

    prev_rows = league_selections[
        (league_selections['team_id'] == team_id) & (league_selections['round'] == gw - 1)
    ]
    if len(prev_rows):
        prev_elements, _ = _position_pool(player_data, position, gw - 1, league_fit['feats'])
        prev_y = np.asarray(ast.literal_eval(prev_rows.iloc[0][POSITION_BINARY_COL[position]]), dtype=float)
        prev_owned = dict(zip(prev_elements.tolist(), prev_y.tolist())) if len(prev_y) == len(prev_elements) else {}
    else:
        prev_owned = {}
    y_prev = np.array([prev_owned.get(e, 0.0) for e in elements])

    alpha = alpha_manager(manager_fit, league_fit, X_raw, y_prev)
    p = alpha / alpha.sum()

    result = pd.DataFrame({
        "element": elements,
        f"owned_gw{gw}": y_actual.astype(int),
        "owned_prev": y_prev.astype(int),
        "p_manager": p,
    }).sort_values("p_manager", ascending=False).reset_index(drop=True)

    owned_ranks = result.index[result[f"owned_gw{gw}"] == 1].to_numpy() + 1
    return result, owned_ranks

GW = 7

gw7_team_id = opponents_by_round[GW]
gw7_results = {}

for position in ['Goalkeeper', 'Defender', 'Midfielder', 'Forward']:
    manager_fit = manager_fits_7[GW][position]
    league_fit = get_league_fit_for_round(LEAGUE_FITS[position], GW)
    result, owned_ranks = evaluate_manager_fit(
        gw7_team_id, position, manager_fit, league_fit, player_data, league_selections, GW
    )
    gw7_results[position] = result
    print(f"{position}: owned ranks {owned_ranks} out of {len(result)}")

gw7_results['Midfielder'].head(10)

Goalkeeper: owned ranks [1 2] out of 70
Defender: owned ranks [1 2 3 4 5] out of 219
Midfielder: owned ranks [1 3 4 5 8] out of 303
Forward: owned ranks [1 2 9] out of 74


,element,owned_gw7,owned_prev,p_manager
0,54,1,1,0.142300
1,317,0,1,0.108519
2,309,1,1,0.106546
3,19,1,1,0.072065
4,328,1,1,0.059148
5,182,0,0,0.002820
6,492,0,0,0.002445
7,491,1,0,0.002306
8,9,0,0,0.002279
9,618,0,0,0.002221


### Flagging Wildcard/Free Hit weeks

Running `evaluate_manager_fit` across every round (below) shows calibration collapsing
on a handful of gameweeks (e.g. GW6, 30, 31, 34, 35) where owned players rank near the
bottom of the pool instead of the top. Checking squad turnover for those rounds shows
11-13 of 15 players changed in a single week, versus 0-2 in the well-calibrated rounds
-- the signature of a Wildcard or Free Hit chip (a near-total squad rebuild with no
transfer-point penalty), not a modelling failure. `alpha_manager`'s retention term
$\gamma_m$ and feature-driven $\Delta\beta_m$ both assume gradual week-to-week
evolution, so they have no signal for a deliberate one-off overhaul.

`detect_squad_overhaul` flags these weeks from squad turnover alone, so downstream code
(e.g. the copula stage) can know not to trust that week's marginals.


In [36]:
def detect_squad_overhaul(team_id, gw, league_selections, threshold=8):
    """
    Flags gameweek `gw` as a likely Wildcard/Free Hit week for `team_id`: a normal
    week's transfers (1-3, more incurring a point hit) change only a handful of the 15
    squad slots, so more than `threshold` changes from gw-1's actual squad is the
    signature of a chip that waives the transfer-point penalty for a full rebuild.

    Returns (n_changed, is_overhaul). Both None if gw-1 or gw isn't in league_selections
    for this team (e.g. gw is the manager's first recorded gameweek).
    """
    prev_rows = league_selections[(league_selections['team_id'] == team_id) & (league_selections['round'] == gw - 1)]
    cur_rows = league_selections[(league_selections['team_id'] == team_id) & (league_selections['round'] == gw)]
    if len(prev_rows) == 0 or len(cur_rows) == 0:
        return None, None

    prev_squad = set(ast.literal_eval(prev_rows.iloc[0]['squad']))
    cur_squad = set(ast.literal_eval(cur_rows.iloc[0]['squad']))
    n_changed = len(cur_squad - prev_squad)
    return n_changed, n_changed > threshold


# sanity check against the rounds identified above
for gw, team in [(6, 23881), (7, 38054), (30, 593532), (34, 2634457)]:
    n_changed, is_overhaul = detect_squad_overhaul(team, gw, league_selections)
    print(f"GW{gw} team {team}: {n_changed} changes, overhaul={is_overhaul}")

GW6 team 23881: 12 changes, overhaul=True
GW7 team 38054: 2 changes, overhaul=False
GW30 team 593532: 11 changes, overhaul=True
GW34 team 2634457: 13 changes, overhaul=True


In [37]:
calibration_records = []
for GW in range(6, 39):
    team_id = opponents_by_round[GW]
    n_changed, is_overhaul = detect_squad_overhaul(team_id, GW, league_selections)
    flag = " [OVERHAUL]" if is_overhaul else ""
    changed_str = n_changed if n_changed is not None else "?"
    print(f"Evaluating manager fits for round {GW} (team {team_id}, {changed_str} squad changes from GW{GW - 1}){flag}...")

    man_fits = joblib.load(f'./estimates/manager_fits_{GW}.joblib')[GW]

    for position in ['Goalkeeper', 'Defender', 'Midfielder', 'Forward']:
        manager_fit = man_fits[position]
        league_fit = get_league_fit_for_round(LEAGUE_FITS[position], GW)
        result, owned_ranks = evaluate_manager_fit(
            team_id, position, manager_fit, league_fit, player_data, league_selections, GW
        )
        print(f"  {position}: owned ranks {owned_ranks} out of {len(result)}")

        calibration_records.append({
            "round": GW,
            "team_id": team_id,
            "position": position,
            "pool_size": len(result),
            "owned_ranks": owned_ranks.tolist(),
            "squad_changes": n_changed,
            "is_overhaul": is_overhaul,
        })

calibration_df = pd.DataFrame(calibration_records)
calibration_df

Evaluating manager fits for round 6 (team 23881, 12 squad changes from GW5) [OVERHAUL]...
  Goalkeeper: owned ranks [62 69] out of 70
  Defender: owned ranks [  1  29  41  81 173] out of 218
  Midfielder: owned ranks [ 15 264 284 290 300] out of 303
  Forward: owned ranks [ 5  7 31] out of 73
Evaluating manager fits for round 7 (team 38054, 2 squad changes from GW6)...
  Goalkeeper: owned ranks [1 2] out of 70
  Defender: owned ranks [1 2 3 4 5] out of 219
  Midfielder: owned ranks [1 3 4 5 8] out of 303
  Forward: owned ranks [1 2 9] out of 74
Evaluating manager fits for round 8 (team 13603, 1 squad changes from GW7)...
  Goalkeeper: owned ranks [1 2] out of 70
  Defender: owned ranks [1 2 4 5 6] out of 219
  Midfielder: owned ranks [  1   3   6   8 267] out of 304
  Forward: owned ranks [1 2 3] out of 74
Evaluating manager fits for round 9 (team 593532, 1 squad changes from GW8)...
  Goalkeeper: owned ranks [1 2] out of 72
  Defender: owned ranks [ 1  2  3  4 50] out of 220
  Midfiel

,round,team_id,position,pool_size,owned_ranks,squad_changes,is_overhaul
0,6,23881,Goalkeeper,70,"[62, 69]",12,True
1,6,23881,Defender,218,"[1, 29, 41, 81, 173]",12,True
2,6,23881,Midfielder,303,"[15, 264, 284, 290, 300]",12,True
3,6,23881,Forward,73,"[5, 7, 31]",12,True
4,7,38054,Goalkeeper,70,"[1, 2]",2,False
...,...,...,...,...,...,...,...
127,37,38054,Forward,87,"[2, 3, 4]",1,False
128,38,13603,Goalkeeper,82,"[1, 2]",0,False
129,38,13603,Defender,268,"[1, 2, 3, 4, 5]",0,False
130,38,13603,Midfielder,347,"[1, 2, 3, 4, 5]",0,False
